# 04 — Temporal Feature Engineering

## Objective

This notebook prepares historical predictors for the future Missed Delivery model. It transforms operational information into features that would have been available **before** the target delivery date.

The analysis in Notebook 03 studied contemporaneous relationships:

$$
X_D \leftrightarrow MD_D
$$

The predictive problem must instead use historical information to predict the outcome on date $D$:

$$
X_{D-k} \rightarrow MD_D
$$

where $k$ is the required time lag and $D-k$ is earlier than the target date.

## Input Data

Two datasets are used:

- `target`: the cleaned analytical dataset, with one observation per `Grupo_raiz` and `Fecha`, including the target outcome `missed_delivery`;
- `history`: the operational history used to construct predictors from previous dates, including production, demand, stock and compliance information.

The first step is to compare their dimensions, date ranges and planning-group coverage. A target observation can only receive a historical feature when the corresponding group has sufficiently available information before its target date.

## Leakage Control

The target date is treated as a strict information boundary. Features must be calculated using data available before $D$, not measurements from the delivery outcome on $D$.

The following outcome or post-event variables must not be used as predictors:

- `lost`
- `delivered`
- `requested`
- `loss_rate`
- `complete_failure`
- `high_volume_md`

Any rolling statistic, lag, encoding or imputation rule must be fitted using the training period only when the data are later split for model evaluation.

## Planned Feature Construction

The feature-engineering workflow will:

1. standardise date and planning-group keys;
2. verify historical coverage and uniqueness at the expected grain;
3. create lagged operational variables;
4. calculate rolling summaries, recent trends and variability measures;
5. add carefully controlled logistical context;
6. quantify missing values created by the temporal look-back; and
7. produce a modelling table with a documented target date and information cut-off for every row.

The final dataset will be evaluated with a chronological split. Random row-level splitting is not suitable as the primary evaluation strategy because it can expose the model to future operational conditions during training.

In [1]:
import pandas as pd

In [2]:
target = pd.read_csv("../../data/processed/dataset_clean.csv")
history = pd.read_csv("../../data/raw/operational_history.csv")

In [3]:
print(target.shape)
print(history.shape)

print(target["Fecha"].min(), target["Fecha"].max())
print(history["Fecha"].min(), history["Fecha"].max())

print(target["Grupo_raiz"].nunique())
print(history["Grupo_raiz"].nunique())

(33971, 27)
(100248, 20)
2025-01-02 2026-08-25
2024-12-29 2026-08-28
226
237


In [4]:
# ============================================================
# HISTORICAL COVERAGE DIAGNOSTIC
# ============================================================

target["Fecha"] = pd.to_datetime(target["Fecha"])
history["Fecha"] = pd.to_datetime(history["Fecha"])

target_groups = set(
    target["Grupo_raiz"].dropna().unique()
)

history_groups = set(
    history["Grupo_raiz"].dropna().unique()
)

missing_in_history = sorted(
    target_groups - history_groups
)

common_groups = (
    target_groups & history_groups
)

print("TARGET")
print("-" * 40)
print(f"Rows:          {len(target):,}")
print(f"Groups:        {len(target_groups)}")
print(
    f"Period:        "
    f"{target['Fecha'].min().date()} -> "
    f"{target['Fecha'].max().date()}"
)

print("\nOPERATIONAL HISTORY")
print("-" * 40)
print(f"Rows:          {len(history):,}")
print(f"Groups:        {len(history_groups)}")
print(
    f"Period:        "
    f"{history['Fecha'].min().date()} -> "
    f"{history['Fecha'].max().date()}"
)

print("\nGROUP COVERAGE")
print("-" * 40)
print(
    f"Target groups with history: "
    f"{len(common_groups)} / {len(target_groups)} "
    f"({len(common_groups) / len(target_groups) * 100:.2f}%)"
)

print(
    f"Groups missing from history: "
    f"{len(missing_in_history)}"
)

print("\nMissing groups:")
print(missing_in_history)


# ============================================================
# TARGET ROWS AFFECTED BY MISSING GROUP HISTORY
# ============================================================

target["has_operational_history"] = (
    target["Grupo_raiz"].isin(history_groups)
)

coverage_by_rows = (
    target["has_operational_history"]
    .value_counts()
    .rename_axis("has_history")
    .reset_index(name="rows")
)

coverage_by_rows["percentage"] = (
    coverage_by_rows["rows"]
    / len(target)
    * 100
)

print("\nTARGET ROW COVERAGE")
print("-" * 40)
print(coverage_by_rows.to_string(index=False))


# ============================================================
# NUMBER OF TARGET OBSERVATIONS BY MISSING GROUP
# ============================================================

missing_group_summary = (
    target.loc[
        ~target["has_operational_history"]
    ]
    .groupby("Grupo_raiz")
    .agg(
        target_rows=("Grupo_raiz", "size"),
        missed_deliveries=(
            "missed_delivery",
            "sum"
        ),
        first_date=("Fecha", "min"),
        last_date=("Fecha", "max")
    )
    .sort_values(
        "target_rows",
        ascending=False
    )
)

print("\nMISSING-GROUP IMPACT")
print("-" * 40)
print(missing_group_summary)

TARGET
----------------------------------------
Rows:          33,971
Groups:        226
Period:        2025-01-02 -> 2026-08-25

OPERATIONAL HISTORY
----------------------------------------
Rows:          100,248
Groups:        237
Period:        2024-12-29 -> 2026-08-28

GROUP COVERAGE
----------------------------------------
Target groups with history: 226 / 226 (100.00%)
Groups missing from history: 0

Missing groups:
[]

TARGET ROW COVERAGE
----------------------------------------
 has_history  rows  percentage
        True 33971       100.0

MISSING-GROUP IMPACT
----------------------------------------
Empty DataFrame
Columns: [target_rows, missed_deliveries, first_date, last_date]
Index: []


## Historical Lag Coverage

The operational-history dataset provides complete planning-group coverage for the target population.

All 226 planning groups present in the target dataset are also represented in the operational history, covering 100% of the 33,971 target observations.

The next step is to evaluate whether operational measurements are available at the exact historical horizons selected for predictive modelling:
$$
D-7,\quad D-14,\quad D-21,\quad D-28
$$
Exact-date coverage is assessed before generating the corresponding lagged features. This prevents assuming that the previous available observation necessarily represents a fixed temporal horizon.

In [5]:
# ============================================================
# EXACT HISTORICAL LAG COVERAGE
# ============================================================

lag_days = [7, 14, 21, 28]

# Unique operational keys available in history
history_keys = (
    history[
        ["Fecha", "Grupo_raiz"]
    ]
    .drop_duplicates()
    .copy()
)

lag_coverage_records = []

for lag in lag_days:

    target_lag = target[
        ["Fecha", "Grupo_raiz"]
    ].copy()

    target_lag["history_date"] = (
        target_lag["Fecha"]
        - pd.Timedelta(days=lag)
    )

    coverage = target_lag.merge(
        history_keys,
        left_on=[
            "history_date",
            "Grupo_raiz"
        ],
        right_on=[
            "Fecha",
            "Grupo_raiz"
        ],
        how="left",
        indicator=True
    )

    available = (
        coverage["_merge"] == "both"
    )

    lag_coverage_records.append({
        "lag_days": lag,
        "target_rows": len(target_lag),
        "rows_with_exact_history": available.sum(),
        "rows_without_exact_history": (~available).sum(),
        "coverage_pct": available.mean() * 100
    })


lag_coverage = pd.DataFrame(
    lag_coverage_records
)

lag_coverage

,lag_days,target_rows,rows_with_exact_history,rows_without_exact_history,coverage_pct
0,7,33971,32860,1111,96.729563
1,14,33971,32356,1615,95.245945
2,21,33971,31876,2095,93.832975
3,28,33971,31442,2529,92.555415


### Exact Lag Coverage Findings

The operational-history dataset provides high exact-date coverage for all selected historical horizons.

The proportion of target observations with an exact historical operational record is:

- D-7: 96.73%
- D-14: 95.25%
- D-21: 93.83%
- D-28: 92.56%

Coverage decreases gradually for longer horizons because earlier target observations have less prior historical information available.

The high coverage obtained at all four horizons supports the use of exact calendar-based lags rather than row-based shifts or approximate nearest-date matching.

Therefore, the temporal feature engineering strategy will use:

$$
X_{D-7},\ X_{D-14},\ X_{D-21},\ X_{D-28}
$$

to predict:

$$
MD_D
$$

## Exact Temporal Lag Construction

The historical coverage analysis confirms that exact operational records are available for more than 92% of target observations even at the longest selected horizon.

Lagged features are therefore constructed using exact calendar offsets rather than row-based shifts.

For each target observation on date $D$, the model-ready dataset retrieves operational information from:

$$
D-7,\quad D-14,\quad D-21,\quad D-28
$$

The resulting feature structure is:

$$
X_{D-7},\ X_{D-14},\ X_{D-21},\ X_{D-28}
\rightarrow MD_D
$$

This approach preserves the real temporal meaning of each lag and avoids assuming that the previous available row corresponds to a fixed number of days.

In [6]:
# ============================================================
# EXACT TEMPORAL LAG FEATURE CONSTRUCTION
# ============================================================

lag_days = [7, 14, 21, 28]

operational_cols = [
    "FabricacionSemana",
    "RitmoSemana",
    "CumpFabRit",
    "ExpedidasSemana",
    "Demanda",
    "CumpExpDem",
    "StockFin",
    "StockReal",
    "CumpStock",
    "AcumFab",
    "AcumRitmo",
    "PorcenCump",
    "AcumEnvio",
    "AcumDemanda",
    "PorcenCumpto",
    "EntregasPrev",
    "DemandActual",
    "CumpTotal"
]

# Keep only the required historical columns
history_lag_base = history[
    ["Fecha", "Grupo_raiz"] + operational_cols
].copy()

# Start from the target dataset
df_features = target.copy()

# ============================================================
# CREATE EXACT D-7 / D-14 / D-21 / D-28 FEATURES
# ============================================================

for lag in lag_days:

    # Historical date required for this horizon
    df_features[f"_history_date_D{lag}"] = (
        df_features["Fecha"]
        - pd.Timedelta(days=lag)
    )

    # Prepare historical dataframe with renamed features
    history_lag = history_lag_base.copy()

    rename_map = {
        col: f"{col}_D{lag}"
        for col in operational_cols
    }

    history_lag = history_lag.rename(
        columns={
            "Fecha": f"_history_date_D{lag}",
            **rename_map
        }
    )

    # Merge by exact historical date + planning group
    df_features = df_features.merge(
        history_lag,
        on=[
            f"_history_date_D{lag}",
            "Grupo_raiz"
        ],
        how="left",
        validate="many_to_one"
    )


# ============================================================
# REMOVE AUXILIARY DATE COLUMNS
# ============================================================

history_date_cols = [
    f"_history_date_D{lag}"
    for lag in lag_days
]

df_features = df_features.drop(
    columns=history_date_cols
)


# ============================================================
# BASIC VALIDATION
# ============================================================

lag_feature_cols = [
    f"{col}_D{lag}"
    for lag in lag_days
    for col in operational_cols
]

print("Original target shape:")
print(target.shape)

print("\nFeature-engineered shape:")
print(df_features.shape)

print("\nNumber of lagged features:")
print(len(lag_feature_cols))

print("\nDuplicate Fecha + Grupo_raiz rows:")
print(
    df_features.duplicated(
        subset=["Fecha", "Grupo_raiz"]
    ).sum()
)

print("\nSample lagged columns:")
print(
    lag_feature_cols[:12]
)

Original target shape:
(33971, 28)

Feature-engineered shape:
(33971, 100)

Number of lagged features:
72

Duplicate Fecha + Grupo_raiz rows:
0

Sample lagged columns:
['FabricacionSemana_D7', 'RitmoSemana_D7', 'CumpFabRit_D7', 'ExpedidasSemana_D7', 'Demanda_D7', 'CumpExpDem_D7', 'StockFin_D7', 'StockReal_D7', 'CumpStock_D7', 'AcumFab_D7', 'AcumRitmo_D7', 'PorcenCump_D7']


## Missingness Introduced by Historical Lags

Exact-date temporal lags naturally introduce missing values when the required historical observation is not available.

This missingness is expected to increase for longer horizons because early target observations have less prior history.

Before modelling, the lag-generated missingness is evaluated to determine:

- the percentage of unavailable historical features at each lag;
- whether missingness is concentrated at the beginning of the observation period;
- whether Missed Delivery and non-Missed Delivery observations are affected differently.

This diagnostic is necessary before deciding whether rows should be filtered, missing values should be preserved, or an explicit availability indicator should be introduced.

In [7]:
# ============================================================
# LAG MISSINGNESS DIAGNOSTIC
# ============================================================

lag_missingness_records = []

for lag in lag_days:

    cols = [
        f"{col}_D{lag}"
        for col in operational_cols
    ]

    # A row is considered available for this lag
    # if at least one lagged operational value exists.
    has_history = (
        df_features[cols]
        .notna()
        .any(axis=1)
    )

    lag_missingness_records.append({
        "lag_days": lag,
        "rows_with_history": has_history.sum(),
        "rows_without_history": (~has_history).sum(),
        "coverage_pct": has_history.mean() * 100,
        "md_rate_with_history": (
            df_features.loc[
                has_history,
                "missed_delivery"
            ].mean() * 100
        ),
        "md_rate_without_history": (
            df_features.loc[
                ~has_history,
                "missed_delivery"
            ].mean() * 100
        )
    })

lag_missingness_summary = pd.DataFrame(
    lag_missingness_records
)

lag_missingness_summary

,lag_days,rows_with_history,rows_without_history,coverage_pct,md_rate_with_history,md_rate_without_history
0,7,32860,1111,96.729563,7.741935,15.391539
1,14,32356,1615,95.245945,7.636914,15.108359
2,21,31876,2095,93.832975,7.569959,14.415274
3,28,31442,2529,92.555415,7.404109,15.302491


In [8]:
# ============================================================
# TEMPORAL LOCATION OF MISSING LAGS
# ============================================================

for lag in lag_days:

    cols = [
        f"{col}_D{lag}"
        for col in operational_cols
    ]

    missing_rows = (
        df_features[cols]
        .isna()
        .all(axis=1)
    )

    if missing_rows.any():

        print(f"\nD-{lag}")
        print("-" * 30)

        print(
            "Missing rows:",
            missing_rows.sum()
        )

        print(
            "First target date with missing history:",
            df_features.loc[
                missing_rows,
                "Fecha"
            ].min()
        )

        print(
            "Last target date with missing history:",
            df_features.loc[
                missing_rows,
                "Fecha"
            ].max()
        )


D-7
------------------------------
Missing rows: 1111
First target date with missing history: 2025-01-02 00:00:00
Last target date with missing history: 2026-08-25 00:00:00

D-14
------------------------------
Missing rows: 1615
First target date with missing history: 2025-01-02 00:00:00
Last target date with missing history: 2026-08-25 00:00:00

D-21
------------------------------
Missing rows: 2095
First target date with missing history: 2025-01-02 00:00:00
Last target date with missing history: 2026-08-25 00:00:00

D-28
------------------------------
Missing rows: 2529
First target date with missing history: 2025-01-02 00:00:00
Last target date with missing history: 2026-08-25 00:00:00


### Lag Missingness Findings

Exact-date historical coverage remains high, ranging from approximately 96.7% at D-7 to 92.6% at D-28.

However, missing lag observations are not restricted to the beginning of the historical period. Missing exact-date records are observed throughout the complete target period.

An additional relevant pattern is observed in the target distribution. Missed Delivery rates among observations without an exact historical record are approximately twice as high as those observed when historical information is available.

Therefore, lag missingness should not be assumed to be completely random.

Rows with unavailable exact historical information will not be removed at this stage. Instead, the availability of historical information will be explicitly represented and the temporal distance to the most recent previous operational observation will be analysed.

This preserves potentially informative missingness while preventing arbitrary imputation.

In [9]:
# ============================================================
# PREVIOUS-AVAILABLE HISTORY DIAGNOSTIC
# ============================================================

target_lookup = (
    target[
        ["Fecha", "Grupo_raiz", "missed_delivery"]
    ]
    .reset_index()
    .rename(columns={"index": "target_index"})
)

history_lookup = (
    history[
        ["Fecha", "Grupo_raiz"]
    ]
    .drop_duplicates()
    .rename(columns={"Fecha": "available_history_date"})
)

history_lookup = history_lookup.sort_values(
    ["available_history_date", "Grupo_raiz"]
)

staleness_records = []

for lag in lag_days:

    lookup = target_lookup.copy()

    lookup["desired_history_date"] = (
        lookup["Fecha"]
        - pd.Timedelta(days=lag)
    )

    # merge_asof requires sorting by merge date
    lookup = lookup.sort_values(
        ["desired_history_date", "Grupo_raiz"]
    )

    matched = pd.merge_asof(
        lookup,
        history_lookup,
        left_on="desired_history_date",
        right_on="available_history_date",
        by="Grupo_raiz",
        direction="backward",
        allow_exact_matches=True
    )

    matched["staleness_days"] = (
        matched["desired_history_date"]
        - matched["available_history_date"]
    ).dt.days

    exact = (
        matched["staleness_days"] == 0
    )

    non_exact = matched.loc[
        ~exact & matched["staleness_days"].notna(),
        "staleness_days"
    ]

    staleness_records.append({
        "lag_days": lag,
        "exact_pct": exact.mean() * 100,
        "previous_available_pct": (
            matched["available_history_date"]
            .notna()
            .mean()
            * 100
        ),
        "median_staleness_non_exact": (
            non_exact.median()
        ),
        "p75_staleness_non_exact": (
            non_exact.quantile(0.75)
        ),
        "p90_staleness_non_exact": (
            non_exact.quantile(0.90)
        ),
        "p95_staleness_non_exact": (
            non_exact.quantile(0.95)
        ),
        "max_staleness_non_exact": (
            non_exact.max()
        )
    })

staleness_summary = pd.DataFrame(
    staleness_records
)

staleness_summary

,lag_days,exact_pct,previous_available_pct,median_staleness_non_exact,p75_staleness_non_exact,p90_staleness_non_exact,p95_staleness_non_exact,max_staleness_non_exact
0,7,96.729563,99.364164,1.0,7.0,12.0,12.0,106.0
1,14,95.245945,98.189632,1.0,2.0,5.0,5.0,105.0
2,21,93.832975,96.685408,1.0,1.0,1.0,2.0,98.0
3,28,92.555415,95.187071,1.0,1.0,2.0,2.0,103.0


### Historical Staleness Assessment

When an exact historical observation is unavailable, the most recent previous operational record is often relatively close to the desired date. However, the staleness distribution contains a non-negligible long tail.

For example, among non-exact D-7 observations, the median staleness is one day, but the 75th and 90th percentiles increase to approximately 7 and 12 days, respectively. Extreme gaps above 100 days are also observed.

Automatically replacing an exact lag with the most recent previous observation could therefore change the temporal meaning of the feature.

A variable such as:

$$
X_{D-7}
$$

should represent information observed exactly seven calendar days before the target date rather than an observation from an uncontrolled earlier date.

For this reason, exact temporal lags are preserved and unavailable historical observations remain missing.

Historical availability is represented explicitly through additional indicator variables.

In [10]:
# ============================================================
# HISTORICAL AVAILABILITY FEATURES
# ============================================================

for lag in lag_days:

    lag_cols = [
        f"{col}_D{lag}"
        for col in operational_cols
    ]

    df_features[f"history_available_D{lag}"] = (
        df_features[lag_cols]
        .notna()
        .any(axis=1)
        .astype(int)
    )


availability_cols = [
    f"history_available_D{lag}"
    for lag in lag_days
]

df_features["available_lag_count"] = (
    df_features[availability_cols]
    .sum(axis=1)
)


# ============================================================
# SUMMARY
# ============================================================

availability_summary = (
    df_features[
        "available_lag_count"
    ]
    .value_counts()
    .sort_index()
    .rename_axis("available_lags")
    .reset_index(name="rows")
)

availability_summary["percentage"] = (
    availability_summary["rows"]
    / len(df_features)
    * 100
)

availability_summary["md_rate"] = (
    availability_summary[
        "available_lags"
    ]
    .map(
        df_features.groupby(
            "available_lag_count"
        )["missed_delivery"]
        .mean()
        * 100
    )
)

availability_summary

,available_lags,rows,percentage,md_rate
0,0,226,0.665273,30.088496
1,1,493,1.451238,20.486815
2,2,647,1.904566,14.374034
3,3,3673,10.812163,9.338415
4,4,28932,85.166760,7.292963


### Historical Availability Pattern

Historical availability itself presents a marked relationship with Missed Delivery occurrence.

Observations with all four exact historical lags available show a Missed Delivery rate of approximately 7.29%, whereas observations with progressively less historical coverage present increasingly higher event rates.

The observed rates range from approximately 9.34% with three available lags to 30.09% when none of the four exact lag dates is available.

This pattern suggests that historical-data availability may contain operational information. However, it should not automatically be interpreted as a causal risk factor.

Missing historical observations may reflect planning-group lifecycle, changes in operational activity, irregular update schedules or other characteristics of the underlying process.

Because availability at $D-k$ is known before the target date $D$, these indicators do not directly constitute target leakage. Nevertheless, their temporal and operational stability must be evaluated before they are retained as predictive features.

In [11]:
# ============================================================
# INVESTIGATE ZERO / LOW HISTORICAL AVAILABILITY
# ============================================================

# Order target observations chronologically inside each group
df_features = df_features.sort_values(
    ["Grupo_raiz", "Fecha"]
).copy()

# Position of each target observation inside its planning group
df_features["group_observation_number"] = (
    df_features
    .groupby("Grupo_raiz")
    .cumcount() + 1
)

availability_detail = (
    df_features
    .groupby("available_lag_count")
    .agg(
        rows=("missed_delivery", "size"),
        md_rate=("missed_delivery", "mean"),
        median_group_observation=(
            "group_observation_number",
            "median"
        ),
        first_date=("Fecha", "min"),
        last_date=("Fecha", "max")
    )
    .reset_index()
)

availability_detail["md_rate"] *= 100

availability_detail

,available_lag_count,rows,md_rate,median_group_observation,first_date,last_date
0,0,226,30.088496,1.0,2025-01-02,2026-08-25
1,1,493,20.486815,4.0,2025-01-06,2026-08-25
2,2,647,14.374034,9.0,2025-01-13,2026-08-24
3,3,3673,9.338415,72.0,2025-01-20,2026-08-20
4,4,28932,7.292963,118.0,2025-01-27,2026-08-06


In [12]:
# Are zero-lag rows simply the first observations of each group?

zero_lag = df_features[
    df_features["available_lag_count"] == 0
]

print("Zero-lag rows:", len(zero_lag))

print(
    "Zero-lag rows that are the first target observation "
    "of their group:",
    (zero_lag["group_observation_number"] == 1).sum()
)

print(
    "Percentage:",
    (
        zero_lag["group_observation_number"] == 1
    ).mean() * 100
)

Zero-lag rows: 226
Zero-lag rows that are the first target observation of their group: 121
Percentage: 53.53982300884957


### Cold-Start and Historical-Support Interpretation

Historical availability is partly related to planning-group maturity.

Among the 226 observations with none of the four exact lags available, approximately 53.5% correspond to the first target observation recorded for their planning group.

However, the remaining observations occur later in the group history, indicating that missing lag information cannot be explained exclusively by cold start.

A clear relationship is observed between historical support and Missed Delivery frequency:

- 0 available lags: 30.09% MD rate;
- 1 available lag: 20.49%;
- 2 available lags: 14.37%;
- 3 available lags: 9.34%;
- 4 available lags: 7.29%.

This pattern suggests that planning-group maturity and continuity of operational history may contain useful predictive information.

Rather than using the position of an observation inside the target dataset, historical-support features will be derived directly from the operational-history dataset so that they represent information genuinely available before date $D$.

In [13]:
# ============================================================
# HISTORICAL SUPPORT / GROUP MATURITY FEATURES
# Fixed datetime precision for merge_asof
# ============================================================

# Ensure consistent datetime precision
df_features["Fecha"] = pd.to_datetime(
    df_features["Fecha"]
).astype("datetime64[ns]")

history["Fecha"] = pd.to_datetime(
    history["Fecha"]
).astype("datetime64[ns]")


# ============================================================
# 1. SORT OPERATIONAL HISTORY
# ============================================================

history_sorted = (
    history
    .sort_values(
        ["Grupo_raiz", "Fecha"]
    )
    .copy()
)


# ============================================================
# 2. FIRST HISTORICAL DATE PER PLANNING GROUP
# ============================================================

first_history_date = (
    history_sorted
    .groupby("Grupo_raiz")["Fecha"]
    .min()
    .rename("first_history_date")
)

df_features = df_features.drop(
    columns=["first_history_date"],
    errors="ignore"
)

df_features = df_features.merge(
    first_history_date,
    on="Grupo_raiz",
    how="left",
    validate="many_to_one"
)


# Days elapsed since first operational record
df_features["days_since_first_history"] = (
    df_features["Fecha"]
    - df_features["first_history_date"]
).dt.days


# ============================================================
# 3. COUNT HISTORICAL RECORDS STRICTLY BEFORE D
# ============================================================

target_temp = (
    df_features[
        ["Fecha", "Grupo_raiz"]
    ]
    .reset_index()
    .rename(
        columns={
            "index": "_row_id"
        }
    )
    .copy()
)


history_temp = (
    history_sorted[
        ["Fecha", "Grupo_raiz"]
    ]
    .drop_duplicates()
    .copy()
)


# Count operational records chronologically
history_temp["history_records_before_D"] = (
    history_temp
    .groupby("Grupo_raiz")
    .cumcount() + 1
)


# Rename history date
history_temp = history_temp.rename(
    columns={
        "Fecha": "_history_date"
    }
)


# Force identical datetime dtype
history_temp["_history_date"] = pd.to_datetime(
    history_temp["_history_date"]
).astype("datetime64[ns]")


# Lookup must be strictly before target date
target_temp["_lookup_date"] = (
    pd.to_datetime(
        target_temp["Fecha"]
    ).astype("datetime64[ns]")
    - pd.Timedelta(nanoseconds=1)
)


# ============================================================
# 4. SORT FOR merge_asof
# ============================================================

target_temp = target_temp.sort_values(
    ["_lookup_date", "Grupo_raiz"]
)

history_temp = history_temp.sort_values(
    ["_history_date", "Grupo_raiz"]
)


# ============================================================
# 5. MATCH MOST RECENT OPERATIONAL RECORD BEFORE D
# ============================================================

support_match = pd.merge_asof(
    target_temp,
    history_temp,
    left_on="_lookup_date",
    right_on="_history_date",
    by="Grupo_raiz",
    direction="backward",
    allow_exact_matches=True
)


support_match["history_records_before_D"] = (
    support_match["history_records_before_D"]
    .fillna(0)
    .astype(int)
)


# Restore original target order
support_match = (
    support_match
    .set_index("_row_id")
    .sort_index()
)


df_features["history_records_before_D"] = (
    support_match[
        "history_records_before_D"
    ].values
)


# ============================================================
# 6. DIAGNOSTIC
# ============================================================

print(
    df_features[
        [
            "days_since_first_history",
            "history_records_before_D",
            "available_lag_count"
        ]
    ].describe()
)


print("\nSpearman correlation:")

print(
    df_features[
        [
            "days_since_first_history",
            "history_records_before_D",
            "available_lag_count"
        ]
    ].corr(
        method="spearman"
    )
)

       days_since_first_history  history_records_before_D  available_lag_count
count              33971.000000              33971.000000         33971.000000
mean                 275.628065                265.880221             3.783639
std                  167.089792                161.067903             0.612012
min                    0.000000                  0.000000             0.000000
25%                  127.000000                121.000000             4.000000
50%                  275.000000                263.000000             4.000000
75%                  418.000000                404.000000             4.000000
max                  604.000000                565.000000             4.000000

Spearman correlation:
                          days_since_first_history  history_records_before_D  \
days_since_first_history                  1.000000                  0.999552   
history_records_before_D                  0.999552                  1.000000   
available_lag_count       

### Historical-Support Findings

The two planning-group maturity measures are almost perfectly correlated.

The Spearman correlation between `days_since_first_history` and `history_records_before_D` is approximately 0.9996, indicating that both variables capture essentially the same historical-support dimension.

To avoid introducing redundant information into the predictive dataset, only `history_records_before_D` is retained as the primary planning-group maturity feature.

In contrast, `available_lag_count` presents a substantially weaker correlation of approximately 0.29 with historical maturity.

This confirms that missing exact lags are not explained exclusively by recently introduced planning groups. Operational-history discontinuities also contribute to lag availability.

The predictive feature set therefore retains both historical maturity and exact-lag availability as distinct types of information.

In [14]:
df_features = df_features.drop(
    columns=[
        "days_since_first_history",
        "first_history_date",
        "group_observation_number"
    ],
    errors="ignore"
)

## Temporal Change and Trend Features

Exact lagged values describe the operational state at specific historical horizons, but they do not directly represent whether the process is improving or deteriorating over time.

To capture recent evolution, additional change features are constructed from the difference between recent and older historical observations.

For a generic operational variable $X$:

$$
\Delta X_{7-28}
=
X_{D-7}
-
X_{D-28}
$$

A positive or negative value therefore represents the direction of change between the two historical horizons.

Shorter-term changes are also calculated:

$$
\Delta X_{7-14}
=
X_{D-7}
-
X_{D-14}
$$

and:

$$
\Delta X_{14-28}
=
X_{D-14}
-
X_{D-28}
$$

These features allow the predictive model to distinguish between a stable operational condition and one that is progressively improving or deteriorating.

In [15]:
trend_base_cols = [
    "CumpStock",
    "stock_gap",
    "CumpExpDem",
    "shipment_demand_gap",
    "CumpFabRit",
    "production_gap",
    "PorcenCumpto",
    "accumulated_delivery_gap",
    "PorcenCump",
    "accumulated_production_gap",
    "CumpTotal",
    "planned_delivery_gap",
    "Demanda",
    "DemandActual",
    "EntregasPrev"
]

In [16]:
# ============================================================
# HISTORICAL GAP FEATURES
# ============================================================

for lag in lag_days:

    df_features[f"production_gap_D{lag}"] = (
        df_features[f"FabricacionSemana_D{lag}"]
        - df_features[f"RitmoSemana_D{lag}"]
    )

    df_features[f"shipment_demand_gap_D{lag}"] = (
        df_features[f"ExpedidasSemana_D{lag}"]
        - df_features[f"Demanda_D{lag}"]
    )

    df_features[f"stock_gap_D{lag}"] = (
        df_features[f"StockReal_D{lag}"]
        - df_features[f"StockFin_D{lag}"]
    )

    df_features[f"accumulated_production_gap_D{lag}"] = (
        df_features[f"AcumFab_D{lag}"]
        - df_features[f"AcumRitmo_D{lag}"]
    )

    df_features[f"accumulated_delivery_gap_D{lag}"] = (
        df_features[f"AcumEnvio_D{lag}"]
        - df_features[f"AcumDemanda_D{lag}"]
    )

    df_features[f"planned_delivery_gap_D{lag}"] = (
        df_features[f"EntregasPrev_D{lag}"]
        - df_features[f"DemandActual_D{lag}"]
    )


# ============================================================
# TEMPORAL CHANGE FEATURES
# ============================================================

trend_base_cols = [
    "CumpStock",
    "stock_gap",
    "CumpExpDem",
    "shipment_demand_gap",
    "CumpFabRit",
    "production_gap",
    "PorcenCumpto",
    "accumulated_delivery_gap",
    "PorcenCump",
    "accumulated_production_gap",
    "CumpTotal",
    "planned_delivery_gap",
    "Demanda",
    "DemandActual",
    "EntregasPrev"
]


for col in trend_base_cols:

    # Short-term change
    df_features[f"{col}_delta_D7_D14"] = (
        df_features[f"{col}_D7"]
        - df_features[f"{col}_D14"]
    )

    # Medium-term change
    df_features[f"{col}_delta_D14_D28"] = (
        df_features[f"{col}_D14"]
        - df_features[f"{col}_D28"]
    )

    # Overall recent trend
    df_features[f"{col}_delta_D7_D28"] = (
        df_features[f"{col}_D7"]
        - df_features[f"{col}_D28"]
    )


# ============================================================
# VALIDATION
# ============================================================

historical_gap_cols = [
    f"{gap}_D{lag}"
    for lag in lag_days
    for gap in [
        "production_gap",
        "shipment_demand_gap",
        "stock_gap",
        "accumulated_production_gap",
        "accumulated_delivery_gap",
        "planned_delivery_gap"
    ]
]

trend_cols = [
    f"{col}_{delta}"
    for col in trend_base_cols
    for delta in [
        "delta_D7_D14",
        "delta_D14_D28",
        "delta_D7_D28"
    ]
]

print("Historical gap features created:")
print(len(historical_gap_cols))

print("\nTemporal trend features created:")
print(len(trend_cols))

print("\nCurrent dataset shape:")
print(df_features.shape)

print("\nSample trend columns:")
print(trend_cols[:12])

Historical gap features created:
24

Temporal trend features created:
45

Current dataset shape:
(33971, 175)

Sample trend columns:
['CumpStock_delta_D7_D14', 'CumpStock_delta_D14_D28', 'CumpStock_delta_D7_D28', 'stock_gap_delta_D7_D14', 'stock_gap_delta_D14_D28', 'stock_gap_delta_D7_D28', 'CumpExpDem_delta_D7_D14', 'CumpExpDem_delta_D14_D28', 'CumpExpDem_delta_D7_D28', 'shipment_demand_gap_delta_D7_D14', 'shipment_demand_gap_delta_D14_D28', 'shipment_demand_gap_delta_D7_D28']


## Temporal Feature Availability

Temporal change features require multiple historical observations to be simultaneously available.

For example:

$$
\Delta X_{7-28}
=
X_{D-7}
-
X_{D-28}
$$

can only be calculated when both $X_{D-7}$ and $X_{D-28}$ are available.

Consequently, derived temporal features may present higher missingness than individual lagged observations.

Before introducing additional features, the availability of the newly created historical gaps and temporal changes is evaluated.

This step prevents unnecessary feature proliferation and provides the information required to define the preprocessing strategy for the predictive models.

In [17]:
# ============================================================
# TEMPORAL FEATURE MISSINGNESS AUDIT
# ============================================================

feature_groups = {
    "exact_lags": lag_feature_cols,
    "historical_gaps": historical_gap_cols,
    "temporal_deltas": trend_cols,
    "availability": [
        "history_available_D7",
        "history_available_D14",
        "history_available_D21",
        "history_available_D28",
        "available_lag_count",
        "history_records_before_D"
    ]
}


# ------------------------------------------------------------
# Summary by feature family
# ------------------------------------------------------------

family_records = []

for family, cols in feature_groups.items():

    existing_cols = [
        col for col in cols
        if col in df_features.columns
    ]

    missing_pct = (
        df_features[existing_cols]
        .isna()
        .mean()
        * 100
    )

    family_records.append({
        "feature_family": family,
        "n_features": len(existing_cols),
        "mean_missing_pct": missing_pct.mean(),
        "median_missing_pct": missing_pct.median(),
        "max_missing_pct": missing_pct.max()
    })


feature_family_missingness = pd.DataFrame(
    family_records
)

print("MISSINGNESS BY FEATURE FAMILY")
print("-" * 50)

display(feature_family_missingness)


# ------------------------------------------------------------
# Individual temporal features
# ------------------------------------------------------------

all_temporal_cols = (
    lag_feature_cols
    + historical_gap_cols
    + trend_cols
)

feature_missingness = (
    df_features[all_temporal_cols]
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("missing_pct")
    .reset_index()
    .rename(columns={"index": "feature"})
)


print("\nFEATURES WITH HIGHEST MISSINGNESS")
print("-" * 50)

display(
    feature_missingness.head(20)
)


# ------------------------------------------------------------
# Features exceeding selected thresholds
# ------------------------------------------------------------

for threshold in [5, 10, 20, 30]:

    n_features = (
        feature_missingness["missing_pct"]
        > threshold
    ).sum()

    print(
        f"Features with > {threshold}% missing: "
        f"{n_features}"
    )


# ------------------------------------------------------------
# Complete temporal-feature rows
# ------------------------------------------------------------

complete_temporal_rows = (
    df_features[all_temporal_cols]
    .notna()
    .all(axis=1)
)

print("\nCOMPLETE TEMPORAL FEATURE MATRIX")
print("-" * 50)

print(
    f"Rows with every temporal feature available: "
    f"{complete_temporal_rows.sum():,}"
)

print(
    f"Percentage: "
    f"{complete_temporal_rows.mean() * 100:.2f}%"
)

print(
    f"MD rate among complete rows: "
    f"{df_features.loc[complete_temporal_rows, 'missed_delivery'].mean() * 100:.2f}%"
)

print(
    f"MD rate among incomplete rows: "
    f"{df_features.loc[~complete_temporal_rows, 'missed_delivery'].mean() * 100:.2f}%"
)

MISSINGNESS BY FEATURE FAMILY
--------------------------------------------------


,feature_family,n_features,mean_missing_pct,median_missing_pct,max_missing_pct
0,exact_lags,72,5.409025,5.460540,7.444585
1,historical_gaps,24,5.409025,5.460540,7.444585
2,temporal_deltas,45,9.033195,9.990875,10.335286
3,availability,6,0.000000,0.000000,0.000000



FEATURES WITH HIGHEST MISSINGNESS
--------------------------------------------------


,feature,missing_pct
0,shipment_demand_gap_delta_D14_D28,10.335286
1,accumulated_delivery_gap_delta_D14_D28,10.335286
2,EntregasPrev_delta_D14_D28,10.335286
3,DemandActual_delta_D14_D28,10.335286
4,stock_gap_delta_D14_D28,10.335286
5,CumpTotal_delta_D14_D28,10.335286
6,planned_delivery_gap_delta_D14_D28,10.335286
7,accumulated_production_gap_delta_D14_D28,10.335286
8,PorcenCump_delta_D14_D28,10.335286
9,production_gap_delta_D14_D28,10.335286


Features with > 5% missing: 93
Features with > 10% missing: 15
Features with > 20% missing: 0
Features with > 30% missing: 0

COMPLETE TEMPORAL FEATURE MATRIX
--------------------------------------------------
Rows with every temporal feature available: 28,932
Percentage: 85.17%
MD rate among complete rows: 7.29%
MD rate among incomplete rows: 12.01%


### Complete vs Incomplete Historical Information

Approximately 85.17% of target observations contain a complete temporal feature set.

However, observations with incomplete historical information present a Missed Delivery rate of approximately 12.01%, compared with 7.29% among observations with complete temporal information.

Therefore, removing incomplete observations would disproportionately exclude higher-risk cases and modify the target distribution.

No global `dropna()` operation is applied.

Missing historical values are preserved together with explicit availability indicators. Numerical imputation, when required by a specific model, will be fitted exclusively on the training data inside the modelling pipeline.

## Leakage-Safe Predictive Feature Set

The feature-engineered dataframe still contains contemporaneous operational variables and delivery-outcome variables inherited from the original analytical dataset.

These variables are useful for retrospective analysis but must not enter the pre-event predictive model.

The predictive dataset is therefore explicitly restricted to:

- logistical context known before the target outcome;
- exact historical operational lags;
- historical gap features;
- temporal change features;
- historical-availability indicators;
- planning-group historical support.

The target outcome and all same-day operational measurements are excluded from the predictive feature matrix.

This ensures that the modelling dataset follows:

$$
X_{D-k} \rightarrow MD_D
$$

rather than using information generated on or after date $D$.

In [18]:
# ============================================================
# LEAKAGE-SAFE PREDICTIVE DATASET
# ============================================================

# ------------------------------------------------------------
# 1. Context variables potentially available before outcome
# ------------------------------------------------------------

context_cols = [
    "Fecha",
    "Grupo_raiz",
    "customer",
    "uat",
    "destination"
]


# ------------------------------------------------------------
# 2. Historical availability / support
# ------------------------------------------------------------

availability_feature_cols = [
    "history_available_D7",
    "history_available_D14",
    "history_available_D21",
    "history_available_D28",
    "available_lag_count",
    "history_records_before_D"
]


# ------------------------------------------------------------
# 3. Historical numerical features
# ------------------------------------------------------------

historical_numeric_cols = (
    lag_feature_cols
    + historical_gap_cols
    + trend_cols
)


# ------------------------------------------------------------
# 4. Predictive feature set
# ------------------------------------------------------------

predictive_feature_cols = (
    context_cols
    + availability_feature_cols
    + historical_numeric_cols
)


# Remove accidental duplicates while preserving order
predictive_feature_cols = list(
    dict.fromkeys(predictive_feature_cols)
)


# ------------------------------------------------------------
# 5. Final modelling dataframe
# ------------------------------------------------------------

model_df = df_features[
    predictive_feature_cols
    + ["missed_delivery"]
].copy()


# ============================================================
# LEAKAGE AUDIT
# ============================================================

forbidden_cols = [
    # Outcome variables
    "lost",
    "delivered",
    "requested",
    "loss_rate",
    "complete_failure",
    "high_volume_md",

    # Same-day operational variables
    "FabricacionSemana",
    "RitmoSemana",
    "CumpFabRit",
    "ExpedidasSemana",
    "Demanda",
    "CumpExpDem",
    "StockFin",
    "StockReal",
    "CumpStock",
    "AcumFab",
    "AcumRitmo",
    "PorcenCump",
    "AcumEnvio",
    "AcumDemanda",
    "PorcenCumpto",
    "EntregasPrev",
    "DemandActual",
    "CumpTotal",

    # Same-day derived gaps
    "production_gap",
    "shipment_demand_gap",
    "stock_gap",
    "accumulated_production_gap",
    "accumulated_delivery_gap",
    "planned_delivery_gap"
]


leakage_found = [
    col
    for col in forbidden_cols
    if col in model_df.columns
]


print("MODEL DATASET")
print("-" * 50)

print(
    f"Rows: {model_df.shape[0]:,}"
)

print(
    f"Columns including target: {model_df.shape[1]}"
)

print(
    f"Predictive variables: {model_df.shape[1] - 1}"
)

print(
    f"Leakage / same-day forbidden columns found: "
    f"{leakage_found}"
)

print(
    "\nTarget distribution:"
)

print(
    model_df["missed_delivery"]
    .value_counts()
)

print(
    "\nTarget rate:"
)

print(
    f"{model_df['missed_delivery'].mean() * 100:.2f}%"
)

MODEL DATASET
--------------------------------------------------
Rows: 33,971
Columns including target: 153
Predictive variables: 152
Leakage / same-day forbidden columns found: []

Target distribution:
missed_delivery
0    31256
1     2715
Name: count, dtype: int64

Target rate:
7.99%


## Feature Redundancy Assessment

The leakage-safe modelling dataset contains 152 candidate predictors.

The purpose of this stage is not to perform aggressive feature selection before modelling, but to identify obvious redundancy introduced during temporal feature engineering.

The assessment focuses on:

- constant or near-constant variables;
- exact duplicate features;
- extremely high pairwise correlations among numerical predictors.

Highly correlated variables are not automatically removed because different modelling algorithms respond differently to redundant information.

Instead, the analysis is used to identify unnecessary duplication and to define a cleaner candidate feature set before model training.

In [19]:
# ============================================================
# FEATURE REDUNDANCY DIAGNOSTIC
# ============================================================

# ------------------------------------------------------------
# 1. Separate metadata / categorical / numerical predictors
# ------------------------------------------------------------

metadata_cols = [
    "Fecha"
]

categorical_predictor_cols = [
    "Grupo_raiz",
    "customer",
    "uat",
    "destination"
]

numeric_predictor_cols = [
    col
    for col in model_df.columns
    if col not in (
        metadata_cols
        + categorical_predictor_cols
        + ["missed_delivery"]
    )
]


print("FEATURE TYPES")
print("-" * 50)

print(
    f"Numeric predictors: "
    f"{len(numeric_predictor_cols)}"
)

print(
    f"Categorical predictors: "
    f"{len(categorical_predictor_cols)}"
)


# ============================================================
# 2. CONSTANT FEATURES
# ============================================================

unique_counts = (
    model_df[numeric_predictor_cols]
    .nunique(dropna=False)
)

constant_features = (
    unique_counts[
        unique_counts <= 1
    ]
    .index
    .tolist()
)


print("\nCONSTANT FEATURES")
print("-" * 50)

print(
    f"Constant features found: "
    f"{len(constant_features)}"
)

print(
    constant_features
)


# ============================================================
# 3. NEAR-CONSTANT FEATURES
# ============================================================

near_constant_records = []

for col in numeric_predictor_cols:

    counts = (
        model_df[col]
        .value_counts(
            normalize=True,
            dropna=False
        )
    )

    if len(counts) > 0:

        dominant_pct = (
            counts.iloc[0]
            * 100
        )

        if dominant_pct >= 99:

            near_constant_records.append({
                "feature": col,
                "dominant_value_pct": dominant_pct,
                "n_unique": model_df[col].nunique(
                    dropna=False
                )
            })


near_constant_features = pd.DataFrame(
    near_constant_records
)


print("\nNEAR-CONSTANT FEATURES (>=99% SAME VALUE)")
print("-" * 50)

display(
    near_constant_features
)


# ============================================================
# 4. EXACT DUPLICATE NUMERICAL FEATURES
# ============================================================

duplicate_pairs = []

for i, col1 in enumerate(
    numeric_predictor_cols
):

    for col2 in numeric_predictor_cols[
        i + 1:
    ]:

        if model_df[col1].equals(
            model_df[col2]
        ):

            duplicate_pairs.append(
                (col1, col2)
            )


print("\nEXACT DUPLICATE FEATURE PAIRS")
print("-" * 50)

print(
    f"Duplicate pairs found: "
    f"{len(duplicate_pairs)}"
)

print(
    duplicate_pairs[:30]
)


# ============================================================
# 5. EXTREMELY HIGH SPEARMAN CORRELATION
# ============================================================

corr_matrix = (
    model_df[numeric_predictor_cols]
    .corr(method="spearman")
    .abs()
)


high_corr_pairs = []

for i in range(
    len(corr_matrix.columns)
):

    for j in range(i):

        corr_value = (
            corr_matrix.iloc[i, j]
        )

        if (
            pd.notna(corr_value)
            and corr_value >= 0.95
        ):

            high_corr_pairs.append({
                "feature_1":
                    corr_matrix.columns[i],

                "feature_2":
                    corr_matrix.columns[j],

                "spearman_abs":
                    corr_value
            })


high_corr_pairs = (
    pd.DataFrame(
        high_corr_pairs
    )
    .sort_values(
        "spearman_abs",
        ascending=False
    )
    if high_corr_pairs
    else pd.DataFrame(
        columns=[
            "feature_1",
            "feature_2",
            "spearman_abs"
        ]
    )
)


print("\nHIGH CORRELATION PAIRS (|rho| >= 0.95)")
print("-" * 50)

print(
    f"Pairs found: "
    f"{len(high_corr_pairs)}"
)

display(
    high_corr_pairs.head(30)
)

FEATURE TYPES
--------------------------------------------------
Numeric predictors: 147
Categorical predictors: 4

CONSTANT FEATURES
--------------------------------------------------
Constant features found: 0
[]

NEAR-CONSTANT FEATURES (>=99% SAME VALUE)
--------------------------------------------------


""



EXACT DUPLICATE FEATURE PAIRS
--------------------------------------------------
Duplicate pairs found: 0
[]

HIGH CORRELATION PAIRS (|rho| >= 0.95)
--------------------------------------------------
Pairs found: 75


,feature_1,feature_2,spearman_abs
73,DemandActual_D28,Demanda_D28,0.990863
4,DemandActual_D7,Demanda_D7,0.990473
20,DemandActual_D14,Demanda_D14,0.990216
45,DemandActual_D21,Demanda_D21,0.989998
52,Demanda_D28,Demanda_D21,0.988548
6,Demanda_D14,Demanda_D7,0.988491
25,Demanda_D21,Demanda_D14,0.987834
58,AcumDemanda_D28,AcumRitmo_D28,0.987110
12,AcumDemanda_D14,AcumRitmo_D14,0.987056
2,AcumDemanda_D7,AcumRitmo_D7,0.986983


In [20]:
# ============================================================
# REVIEW HIGHEST CORRELATED FEATURE PAIRS
# ============================================================

pd.set_option(
    "display.max_rows",
    100
)

display(
    high_corr_pairs.head(75)
)

,feature_1,feature_2,spearman_abs
73,DemandActual_D28,Demanda_D28,0.990863
4,DemandActual_D7,Demanda_D7,0.990473
20,DemandActual_D14,Demanda_D14,0.990216
45,DemandActual_D21,Demanda_D21,0.989998
52,Demanda_D28,Demanda_D21,0.988548
6,Demanda_D14,Demanda_D7,0.988491
25,Demanda_D21,Demanda_D14,0.987834
58,AcumDemanda_D28,AcumRitmo_D28,0.987110
12,AcumDemanda_D14,AcumRitmo_D14,0.987056
2,AcumDemanda_D7,AcumRitmo_D7,0.986983


In [21]:
# ============================================================
# FEATURES MOST INVOLVED IN HIGH CORRELATION
# ============================================================

corr_feature_counts = pd.concat(
    [
        high_corr_pairs["feature_1"],
        high_corr_pairs["feature_2"]
    ]
).value_counts()

corr_feature_counts = (
    corr_feature_counts
    .rename_axis("feature")
    .reset_index(name="high_corr_connections")
)

display(
    corr_feature_counts.head(30)
)

,feature,high_corr_connections
0,DemandActual_D21,11
1,Demanda_D14,11
2,Demanda_D21,11
3,EntregasPrev_D21,11
4,DemandActual_D7,10
5,DemandActual_D14,10
6,Demanda_D28,10
7,EntregasPrev_D14,10
8,Demanda_D7,10
9,DemandActual_D28,9


### Feature Redundancy Findings

No constant, near-constant, or exactly duplicated predictors were identified.

However, 75 numerical feature pairs presented an absolute Spearman correlation greater than or equal to 0.95.

Most of these relationships are concentrated in operational-demand variables such as `Demanda`, `DemandActual`, and `EntregasPrev`, as well as repeated measurements of the same variable across different historical horizons.

This level of correlation is expected because adjacent operational snapshots describe closely related states of the same planning process.

Highly correlated features are not removed at this stage.

In particular, temporal versions of the same variable are intentionally preserved so that the modelling stage can determine whether predictive information decreases as the temporal distance from the Missed Delivery increases.

Model-specific regularisation and feature-importance analyses will subsequently be used to evaluate redundant predictors.

## Calendar Features

Calendar information associated with the scheduled target date is known in advance and can therefore be safely used for prediction.

Three calendar variables are derived:

- month of the year;
- day of the week;
- ISO week of the year.

These variables may capture recurring operational patterns without using any information generated by the delivery outcome itself.

The original date is retained exclusively for temporal splitting and model evaluation and is not used directly as a numerical predictor.

In [22]:
# ============================================================
# CALENDAR FEATURES
# ============================================================

model_df["month"] = (
    model_df["Fecha"]
    .dt.month
    .astype(int)
)

model_df["day_of_week"] = (
    model_df["Fecha"]
    .dt.dayofweek
    .astype(int)
)

model_df["week_of_year"] = (
    model_df["Fecha"]
    .dt.isocalendar()
    .week
    .astype(int)
)


# ============================================================
# FINAL NOTEBOOK 4 VALIDATION
# ============================================================

print("FINAL FEATURE-ENGINEERED DATASET")
print("-" * 50)

print(f"Rows: {model_df.shape[0]:,}")
print(f"Columns: {model_df.shape[1]}")
print(
    f"Predictive variables excluding Fecha and target: "
    f"{model_df.shape[1] - 2}"
)

print("\nDate range:")
print(
    model_df["Fecha"].min(),
    "->",
    model_df["Fecha"].max()
)

print("\nTarget rate:")
print(
    f"{model_df['missed_delivery'].mean() * 100:.2f}%"
)

print("\nCalendar features:")
print(
    model_df[
        [
            "month",
            "day_of_week",
            "week_of_year"
        ]
    ].describe()
)

FINAL FEATURE-ENGINEERED DATASET
--------------------------------------------------
Rows: 33,971
Columns: 156
Predictive variables excluding Fecha and target: 154

Date range:
2025-01-02 00:00:00 -> 2026-08-25 00:00:00

Target rate:
7.99%

Calendar features:
              month   day_of_week  week_of_year
count  33971.000000  33971.000000  33971.000000
mean       5.460834      2.078008     22.370345
std        3.222824      1.430865     13.903034
min        1.000000      0.000000      1.000000
25%        3.000000      1.000000     11.000000
50%        5.000000      2.000000     21.000000
75%        7.000000      3.000000     31.000000
max       12.000000      6.000000     52.000000


## Final Feature-Engineering Dataset

The final feature-engineered dataset contains 33,971 observations and 154 predictive variables, excluding the target and the original date used for temporal splitting.

The final feature space combines:

- logistical context variables;
- exact operational lags at D-7, D-14, D-21, and D-28;
- historical operational gap features;
- temporal change features;
- historical-availability indicators;
- historical-support information;
- calendar variables derived from the scheduled target date.

No same-day operational measurements or delivery-outcome variables are included in the predictive feature set.

The resulting dataset therefore preserves the intended predictive structure:

$$
X_{D-k} \rightarrow MD_D
$$

Missing historical values are intentionally retained and will be handled inside the modelling pipeline using training-only preprocessing.

In [23]:
# ============================================================
# EXPORT FEATURE-ENGINEERED DATASET
# ============================================================

output_path = "../../data/processed/dataset_features.csv"

model_df.to_csv(
    output_path,
    index=False
)

print(f"Dataset exported to: {output_path}")
print(f"Shape: {model_df.shape}")

Dataset exported to: ../../data/processed/dataset_features.csv
Shape: (33971, 156)
